# Chapter 26 — Debugging Context for Coding Agents

**Book alignment:** Debugging AI From First Principles, Chapter 26

**Question this notebook isolates:** The contract is perfect; the agent edits the wrong
module, duplicates an existing helper, and misses the migration. Does the working-set dump
— sent-bytes hash vs current hash vs absent, *per file* — separate **H1** (missing-file,
below top-k), **H2** (misread-file, present + current + ignored), and **H3** (stale context,
older revision)? And is the verdict per-file or per-session?

In [ ]:
import hashlib
def h(s): return hashlib.sha1(s.encode()).hexdigest()[:4]

# current repo state
REPO = {
    "db/migrate_042.py": "add refund_id column",
    "lib/helpers.py":    "def rate_key(user): ...",
    "api/routes.py":     "router v2: /users, /refunds",
}
CURRENT = {p: h(txt) for p, txt in REPO.items()}

# what the agent's session actually sent (from the log): path -> (sent_hash, retrieval_rank)
TOP_K = 5
SENT = {
    "lib/helpers.py":  (h("def rate_key(user): ..."), 2),          # present + current
    "api/routes.py":   (h("router v1: /users"),       3),          # present but STALE hash
    # db/migrate_042.py: never sent (retrieval rank 47, below top-k 5)
}
RETRIEVAL_RANK = {"db/migrate_042.py": 47}
REQUIRED = ["db/migrate_042.py", "lib/helpers.py", "api/routes.py"]

## 1. Build the working-set dump — one row, one verdict, per file

In [ ]:
print(f"{'file':22} {'sent':>6} {'current':>8} {'rank':>5}  verdict")
verdicts = {}
for p in REQUIRED:
    sent = SENT.get(p)
    cur = CURRENT[p]
    if sent is None:
        rank = RETRIEVAL_RANK.get(p, 999)
        v = f"H1 missing-file (rank {rank} > top-k {TOP_K})"
    elif sent[0] != cur:
        v = f"H3 stale-context (sent {sent[0]} != current {cur})"
    else:
        v = "H2 misread-candidate (present + current + still ignored)"
    verdicts[p] = v
    print(f"{p:22} {str(sent[0]) if sent else '--':>6} {cur:>8} "
          f"{str(sent[1]) if sent else '--':>5}  {v}")

assert verdicts["db/migrate_042.py"].startswith("H1")
assert verdicts["api/routes.py"].startswith("H3")
assert verdicts["lib/helpers.py"].startswith("H2")
print("\nthree files, three different gaps, ONE session -> context failure is per-file, not per-session")

## 2. A session-level verdict would mistarget all three repairs

In [ ]:
session_verdict = "the agent can't read code"
per_file_repairs = {
    "db/migrate_042.py": "fix retrieval: raise top-k / expand the dependency-graph frontier",
    "api/routes.py":     "fix cache invalidation: re-index to the current revision",
    "lib/helpers.py":    "surface + reorder: move the helper to context head (position, not comprehension)",
}
assert len(set(per_file_repairs.values())) == 3
print(f"'{session_verdict}' -> one repair for three unrelated gaps")
for p, r in per_file_repairs.items():
    print(f"  {p:22} {r}")

## 3. The single-variable intervention: one file, one change

In [ ]:
# H1 probe: add ONLY the migration to context (everything else fixed) -> agent applies it
def agent_fixes_migration(migration_in_context):
    return migration_in_context            # deterministic: it needs the file to touch it

assert agent_fixes_migration(True) and not agent_fixes_migration(False)
print("add the migration alone -> fix lands. this is the only unconfounded next step for that row.")
print("a paste-everything repair confounds content + position + working-set size.")

## What we earned

Every agent edit is rational relative to *some* working set; reconstruct that set before
judging the decision. The dump — sent-bytes hash vs current hash vs absent, with retrieval
rank — gave three different verdicts for three files in one session: **H1** (migration never
sent, rank 47), **H3** (routes sent at a stale hash), **H2** (helper present, current, and
still ignored — usually position, not comprehension). A session-level "the agent can't
read" would have mistargeted all three repairs.

**Notebook 27 / Chapter 27** takes the case where the agent saw everything and still
proposed a design that cannot meet its budget.